# Práctica 2: Aprendizaje No Supervisado
## Determinación de Tipos de Estrellas

**Asignatura:** Aprendizaje Automático — Universidad Carlos III de Madrid  
**Curso:** 2024–2025

| Nombre | NIA |
|--------|-----|
| Jorge López Alonso | 100495876 |
| Álvaro Carrasco Fuentes | 100495918 |

---

## Objetivo

Aplicar técnicas de aprendizaje no supervisado sobre un dataset de 240 estrellas con el fin de identificar agrupaciones naturales. Se utilizarán tres algoritmos de clustering (K-Means, Clustering Jerárquico y DBSCAN) precedidos de una reducción de dimensionalidad mediante PCA. Los resultados se compararán con las clases astronómicas reales.

**Semilla aleatoria base:** `100495876` (NIA de Jorge López Alonso)

## 1. Importaciones y Configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

from scipy.cluster.hierarchy import dendrogram, linkage

import warnings
warnings.filterwarnings('ignore')

# Semilla aleatoria base (NIA del grupo)
SEED = 100495876
np.random.seed(SEED)

# Estilo de gráficas
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.figsize'] = (10, 6)

print('Librerías cargadas correctamente.')
print(f'Semilla aleatoria: {SEED}')

## 2. Carga y Exploración del Dataset (EDA)

In [ ]:
df = pd.read_csv('../data/stars_data.csv')
print(f'Dimensiones del dataset: {df.shape}')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

### 2.1 Variables Categóricas

In [ ]:
print('Valores únicos en Color:')
print(df['Color'].value_counts().to_string(), '\n')
print('Valores únicos en Spectral_Class:')
print(df['Spectral_Class'].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

color_counts = df['Color'].value_counts()
axes[0].barh(color_counts.index, color_counts.values, color='steelblue', edgecolor='black')
axes[0].set_title('Distribución de Color')
axes[0].set_xlabel('Frecuencia')

spectral_counts = df['Spectral_Class'].value_counts()
axes[1].bar(spectral_counts.index, spectral_counts.values, color='coral', edgecolor='black')
axes[1].set_title('Distribución de Spectral_Class')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

### 2.2 Distribuciones de Variables Numéricas

> **Nota**: Las variables `L` y `R` presentan rangos de varios órdenes de magnitud (ej. L va de 1.38×10⁻⁴ a 3.4×10⁵). Se visualizan tanto en escala lineal como logarítmica.

In [ ]:
num_cols = ['Temperature', 'L', 'R', 'A_M']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i, col in enumerate(num_cols):
    # Escala lineal
    axes[0, i].hist(df[col], bins=30, edgecolor='black', alpha=0.75, color='steelblue')
    axes[0, i].set_title(f'{col} (lineal)')
    axes[0, i].set_xlabel(col)
    axes[0, i].set_ylabel('Frecuencia')

    # Escala logarítmica (solo para valores > 0)
    data_log = df[col][df[col] > 0]
    axes[1, i].hist(np.log10(data_log), bins=30, edgecolor='black', alpha=0.75, color='darkorange')
    axes[1, i].set_title(f'{col} (log₁₀)')
    axes[1, i].set_xlabel(f'log₁₀({col})')
    axes[1, i].set_ylabel('Frecuencia')

plt.suptitle('Distribuciones de variables numéricas (lineal vs. log₁₀)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 2.3 Detección de Outliers (Boxplots)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, col in zip(axes, num_cols):
    ax.boxplot(df[col], patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='navy'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(f'Boxplot {col}')
    ax.set_ylabel(col)

plt.suptitle('Detección de outliers en variables numéricas', fontsize=13)
plt.tight_layout()
plt.show()

# Rango intercuartílico (IQR) para cuantificar outliers
print('Outliers por variable (método IQR):')
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f'  {col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)')

### 2.4 Matriz de Correlación

In [ ]:
# Correlación sobre los datos originales (sin encoding aún)
corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Matriz de correlación (variables numéricas)')
plt.tight_layout()
plt.show()

print('\nCorrelaciones más altas (|r| > 0.5):')
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.5:
            print(f'  {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: r = {r:.3f}')

### 2.5 Scatter Matrix (pairplot)

Visualización de las relaciones entre pares de variables numéricas. Se observa claramente la separación natural entre grupos de estrellas.

In [ ]:
# Usamos log para L y R por su gran rango dinámico
df_plot = df[num_cols].copy()
df_plot['log_L'] = np.log10(df['L'].clip(lower=1e-10))
df_plot['log_R'] = np.log10(df['R'].clip(lower=1e-10))

plot_cols = ['Temperature', 'log_L', 'log_R', 'A_M']
g = sns.pairplot(df_plot[plot_cols], diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20})
g.figure.suptitle('Scatter matrix (L y R en escala log₁₀)', y=1.02, fontsize=13)
plt.show()

**Observaciones del EDA:**
- `L` y `R` tienen rangos de varios órdenes de magnitud → necesitan normalización (StandardScaler).
- `Temperature` y `A_M` muestran distribuciones multimodales, indicio de grupos naturales.
- Alta correlación entre `Temperature` y `L` (estrellas más calientes son más luminosas).
- Los boxplots muestran muchos outliers estadísticos, pero son estrellas físicamente reales (gigantes, enanas blancas).
- El pairplot revela al menos 2–3 grupos visuales bien separados.

---
## 3. Preprocesamiento

### 3.1 Codificación Ordinal de Variables Categóricas

Las variables categóricas deben codificarse respetando el orden físico, **no** alfabéticamente:

- **`Spectral_Class`**: secuencia astronómica de mayor a menor temperatura: `O → B → A → F → G → K → M`
- **`Color`**: de mayor a menor energía (longitud de onda): `Azul → Azul-blanco → Blanco → Blanco-amarillo → Amarillento → Amarillo-naranja pálido → Naranja → Naranja-rojo → Rojo`

> El dataset contiene variantes de texto inconsistentes en `Color` (mayúsculas, guiones, espacios) que se normalizan antes de codificar.

In [ ]:
# --- Normalización de la columna Color ---
# Paso 1: convertir a minúsculas y eliminar espacios extremos
df['Color_norm'] = df['Color'].str.strip().str.lower()

# Paso 2: unificar variantes (guiones, espacios, mayúsculas mezcladas)
color_map = {
    # Azul-blanco (todas las variantes)
    'blue white'       : 'blue-white',
    'blue white '      : 'blue-white',
    'blue-white'       : 'blue-white',
    # Blanco (todas las variantes)
    'white'            : 'white',
    'whitish'          : 'white',
    # Blanco-amarillo (todas las variantes)
    'white-yellow'     : 'white-yellow',
    'yellow-white'     : 'white-yellow',
    # Amarillento (todas las variantes)
    'yellowish white'  : 'yellowish',
    'yellowish'        : 'yellowish',
    # El resto ya tiene forma canónica: 'blue', 'pale yellow orange', 'orange', 'orange-red', 'red'
}
df['Color_norm'] = df['Color_norm'].replace(color_map)

print('Valores de Color tras normalización:')
print(df['Color_norm'].value_counts().to_string())

In [ ]:
# --- Definición del orden ordinal ---
# Spectral_Class: de mayor a menor temperatura (O=0, M=6)
spectral_order = ['O', 'B', 'A', 'F', 'G', 'K', 'M']

# Color: de mayor a menor energía/temperatura (azul=0, rojo=8)
color_order = [
    'blue',
    'blue-white',
    'white',
    'white-yellow',
    'yellowish',
    'pale yellow orange',
    'orange',
    'orange-red',
    'red'
]

# Verificar que todos los valores del dataset están cubiertos
valores_color = set(df['Color_norm'].unique())
no_cubiertos = valores_color - set(color_order)
if no_cubiertos:
    print(f'AVISO: valores sin mapear → {no_cubiertos}')
else:
    print('Todos los valores de Color están correctamente mapeados.')

# --- Aplicar OrdinalEncoder ---
df_enc = df.copy()

enc_spectral = OrdinalEncoder(categories=[spectral_order])
enc_color    = OrdinalEncoder(categories=[color_order])

df_enc['Spectral_Class_enc'] = enc_spectral.fit_transform(df[['Spectral_Class']]).astype(int)
df_enc['Color_enc']          = enc_color.fit_transform(df[['Color_norm']]).astype(int)

# Mostrar tabla de correspondencia
print('\nCorrespondencia Spectral_Class → código:')
for i, v in enumerate(spectral_order):
    print(f'  {v} → {i}')

print('\nCorrespondencia Color → código:')
for i, v in enumerate(color_order):
    print(f'  {v} → {i}')

In [ ]:
# Dataset final para modelado (solo columnas numéricas + encodings)
feature_cols = ['Temperature', 'L', 'R', 'A_M', 'Color_enc', 'Spectral_Class_enc']
X = df_enc[feature_cols].copy()

print('Dataset listo para modelado:')
print(X.head(10).to_string())
print(f'\nShape: {X.shape}')

### 3.2 Normalización con StandardScaler

La normalización es obligatoria: variables como `L` tienen valores del orden de 10⁵ mientras que `Color_enc` va de 0 a 8. Sin escalar, la distancia euclidiana estaría dominada por `L` y `R`.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Verificación: media ≈ 0, std ≈ 1 en cada columna
df_scaled_check = pd.DataFrame(X_scaled, columns=feature_cols)
print('Estadísticos tras StandardScaler (media y std por columna):')
print(df_scaled_check.agg(['mean', 'std']).round(4).to_string())

### 3.3 Reducción de Dimensionalidad: PCA a 2 Componentes

El enunciado indica aplicar los algoritmos de clustering directamente sobre las 2 componentes principales.

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

var_exp = pca.explained_variance_ratio_
print(f'Varianza explicada por PC1: {var_exp[0]*100:.2f}%')
print(f'Varianza explicada por PC2: {var_exp[1]*100:.2f}%')
print(f'Varianza acumulada (PC1+PC2): {sum(var_exp)*100:.2f}%')

print('\nCargas (loadings) de cada variable en PC1 y PC2:')
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_cols,
    columns=['PC1', 'PC2']
).round(3)
print(loadings.to_string())

In [ ]:
# Gráfico de varianza explicada acumulada (scree plot)
pca_full = PCA(random_state=SEED).fit(X_scaled)
var_acum = np.cumsum(pca_full.explained_variance_ratio_) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scree plot
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_)+1),
            pca_full.explained_variance_ratio_*100,
            color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Varianza explicada (%)')
axes[0].set_title('Scree plot')
axes[0].set_xticks(range(1, len(pca_full.explained_variance_ratio_)+1))

# Varianza acumulada
axes[1].plot(range(1, len(var_acum)+1), var_acum, 'o-', color='darkorange', linewidth=2)
axes[1].axhline(y=80, color='gray', linestyle='--', label='80%')
axes[1].axhline(y=90, color='red',  linestyle='--', label='90%')
axes[1].axvline(x=2, color='steelblue', linestyle=':', label='n=2 (enunciado)')
axes[1].set_xlabel('Número de componentes')
axes[1].set_ylabel('Varianza explicada acumulada (%)')
axes[1].set_title('Varianza acumulada')
axes[1].set_xticks(range(1, len(var_acum)+1))
axes[1].legend()

plt.suptitle('Análisis de componentes principales', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot de los datos proyectados en PC1 y PC2
# Coloreamos por Spectral_Class para validar visualmente
spectral_labels = df['Spectral_Class'].values
spectral_classes = spectral_order
colors_sc = plt.cm.tab10(np.linspace(0, 1, len(spectral_classes)))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Plot 1: coloreado por Spectral_Class ---
for i, sc in enumerate(spectral_classes):
    mask = spectral_labels == sc
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    color=colors_sc[i], label=sc, s=40, alpha=0.8, edgecolors='k', linewidths=0.3)
axes[0].set_xlabel(f'PC1 ({var_exp[0]*100:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({var_exp[1]*100:.1f}% var)')
axes[0].set_title('Proyección PCA coloreada por Spectral_Class')
axes[0].legend(title='Spectral_Class', bbox_to_anchor=(1.01, 1))

# --- Plot 2: coloreado por Color normalizado ---
color_labels_norm = df_enc['Color_norm'].values
color_palette = {
    'blue'              : 'royalblue',
    'blue-white'        : 'deepskyblue',
    'white'             : 'lightgray',
    'white-yellow'      : 'khaki',
    'yellowish'         : 'gold',
    'pale yellow orange': 'moccasin',
    'orange'            : 'darkorange',
    'orange-red'        : 'tomato',
    'red'               : 'firebrick'
}
for color_name, hex_color in color_palette.items():
    mask = color_labels_norm == color_name
    if mask.any():
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        color=hex_color, label=color_name, s=40, alpha=0.85,
                        edgecolors='k', linewidths=0.3)
axes[1].set_xlabel(f'PC1 ({var_exp[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({var_exp[1]*100:.1f}% var)')
axes[1].set_title('Proyección PCA coloreada por Color')
axes[1].legend(title='Color', bbox_to_anchor=(1.01, 1), fontsize=8)

plt.suptitle('Espacio PCA 2D — datos sin etiquetar listos para clustering', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nDatos para clustering: X_pca.shape = {X_pca.shape}')
print('Los algoritmos de clustering se aplicarán sobre X_pca (PC1, PC2).')

**Conclusiones del preprocesamiento y PCA:**
- Las 2 primeras componentes explican la mayor parte de la varianza del dataset.
- PC1 está fuertemente relacionada con luminosidad (`L`) y radio (`R`), que distinguen estrellas gigantes de enanas.
- PC2 captura la temperatura y clase espectral, separando estrellas frías (rojas) de calientes (azules).
- La proyección 2D ya muestra grupos visualmente bien diferenciados, lo que presagia buenos resultados de clustering.
- Los datos proyectados `X_pca` se utilizarán como entrada en las secciones 4, 5 y 6.

---
## 4. K-Means Clustering

### 4.1 Selección del número óptimo de clusters (método del codo + Silhouette)

In [ ]:
# TODO: Iterar sobre k=2..10
# TODO: Calcular inercia (método del codo) y Silhouette Score
# TODO: Visualizar curvas y justificar k óptimo

pass

### 4.2 Modelo final K-Means

In [ ]:
# TODO: Entrenar KMeans con k óptimo y random_state=SEED
# TODO: Visualizar clusters sobre PCA
# TODO: Calcular y reportar métricas: Silhouette, Davies-Bouldin, Calinski-Harabasz

pass

---
## 5. Clustering Jerárquico / Dendrogramas

### 5.1 Análisis del dendrograma

In [ ]:
# TODO: Probar distintas funciones de enlace: ward, complete, average, single
# TODO: Visualizar dendrogramas
# TODO: Justificar elección de linkage y número de clusters

pass

### 5.2 Modelo final Jerárquico

In [ ]:
# TODO: Entrenar AgglomerativeClustering con parámetros óptimos
# TODO: Visualizar clusters sobre PCA
# TODO: Calcular métricas: Silhouette, Davies-Bouldin

pass

---
## 6. DBSCAN

### 6.1 Selección de hiperparámetros (eps, min_samples)

In [ ]:
# TODO: Usar heurística k-distancia para estimar eps
# TODO: Grid search sobre (eps, min_samples) evaluando con DBCV
# Nota: DBCV no está en sklearn, instalar con: pip install hdbscan (incluye dbcv)

pass

### 6.2 Modelo final DBSCAN

In [ ]:
# TODO: Entrenar DBSCAN con parámetros óptimos
# TODO: Visualizar clusters sobre PCA (incluir puntos ruido en color gris)
# TODO: Reportar DBCV y número de clusters/ruido encontrados

pass

---
## 7. Comparativa de Algoritmos

In [ ]:
# TODO: Tabla resumen con métricas de los 3 algoritmos
# TODO: Visualización comparativa de los 3 clustering sobre PCA
# TODO: Justificación del algoritmo recomendado

pass

---
## 8. Comparación con Clases Astronómicas

Tabla de referencia de tipos estelares:

| Tipo | T (K) | L/L☉ | R/R☉ | A_M | Color | Clase Espectral |
|------|-------|-------|------|-----|-------|------------------|
| Enana roja | 3000 | 7.0×10⁻⁴ | 1.0×10⁻¹ | +17.5 | Rojo | K-M |
| Enana marrón | 3300 | 5.5×10⁻³ | 3.5×10⁻¹ | +12.5 | Rojo | M |
| Enana blanca | 14000 | 2.5×10⁻³ | 1.0×10⁻² | +12.6 | Blanco | B-G |
| Secuencia principal | 16000 | 3.2×10⁴ | 4.4 | −0.4 | Blanco-amarillo | B-M |
| Supergigante | 15000 | 3.0×10⁵ | 5.0×10¹ | −6.4 | Blanco-amarillo | B-M |
| Hipergigante | 11000 | 3.0×10⁵ | 1.4×10³ | −9.6 | Amarillo | B-M |

In [ ]:
# TODO: Analizar si los clusters obtenidos se corresponden con los tipos astronómicos
# TODO: Calcular centroides de cada cluster y comparar con la tabla de referencia
# TODO: Discutir similitudes y diferencias

pass

---
## 9. Conclusiones

_TODO: Redactar conclusiones sobre:_
- _Algoritmo recomendado y justificación_
- _Correspondencia con clases astronómicas_
- _Limitaciones y posibles mejoras_